# <font color="#003660">Applied Machine Learning for Text Analysis (M.184.5331)</font>


# <font color="#003660">Session 6: LLM Agents with LangChain</font>

# <font color="#003660">From Simple Agents to Multi-Agent Systems</font>

<center><br><img width=256 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/dag.png"/><br></center>

<p>

<div>
    <font color="#085986"><b>By the end of this lesson, you ...</b><br><br>
        ... will know how LLM agents are build with LangChain and LangGraph. <br>
        ... will know how Multi-Agent Systems with the controller architecture are build with LangChain.
    </font>
</div>
</p>

The following content is heavily inspired by the following excellent sources:

* [LangChain Academy](https://academy.langchain.com/)
* [Introduction to LangChain Agents](https://github.com/langchain-ai/langchain-academy/blob/main/module-1/agent.ipynb)
* [LangChain Docs (Python)](https://python.langchain.com/)

In [ ]:
!pip install -U wikipedia langchain langchain-community langchain-openai

Today we will setup our own ollama server. We can do this directly in Google Colab.

First we need to install the pciutil package (to let ollama automatically detect GPU) and ollama. Just run the code below.

In [ ]:
# with this linux package, ollama can then detect GPU, if available
!sudo apt-get install -y pciutils

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

The next chunk is to start ollama locally as a subprocess in the background. (Even if ollama tells you that it has started the server, it has not.)

In [16]:
import subprocess

process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)


Now we will have to download both models for this session. Run the code below.

In [ ]:
!ollama pull qwen3:8b # takes around a minute

## Answering Questions using LLMs

In [ ]:
from langchain_community.retrievers import WikipediaRetriever

from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

Today we will use Qwen3, a reasoning LLM from Alibaba ([Yang et al., 2025](https://doi.org/10.48550/arXiv.2505.09388)). These are special LLMs trained to first reason step-by-step similar to Chain-of-Thought you learned last session. This training approach shifts LLMs away from fast, error-prone (System 1) responses toward more deliberate, reflective (System 2) reasoning about the task ([Li et a., 2025](https://doi.org/10.48550/arXiv.2502.17419)).

This is especially helpful when we want to do synthesis of information.

In [ ]:
@tool
def search_in_wikipedia(query: str) -> str:
    """Search Wikipedia for a given query."""
    retriever = WikipediaRetriever()
    docs = retriever.invoke(query)
    results = "\n\n-----\n\n".join([f"Document {i}:\n\nMetadata:\n-Title: {docs[i].metadata['title'].strip()}\n-Path: https://en.wikipedia.org/wiki/{docs[i].metadata['title'].strip().replace(' ', '_')}\n\nContent:\n{docs[i].metadata['summary'].strip()}" for i in range(len(docs))])

    return results

tools = [
    search_in_wikipedia
]

In [ ]:
SYSTEM_PROMPT = """You are a knowledgeable and helpful assistant that answers user queries.
You are able to retrieve and accurate information from Wikipedia."""

config = {
    "model": "qwen3:8b",
    "base_url": "http://127.0.0.1:11434/v1",
	"api_key": "ollama",
    "max_tokens": 8192,
    "temperature": 0,
    "seed": 42,
}

model = ChatOpenAI(
    **config
)
react_agent = create_agent(
    model,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)

In [ ]:
query = "Who is Daenerys Targaryen and who is her actress?"
messages = [HumanMessage(content=query)]
messages = react_agent.invoke({"messages": messages})["messages"]
for message in messages:
    print(message.pretty_repr())

## Build your own agent (A Blueprint)

In [ ]:
@tool
def your_tool_here(your_variables_here):
    # ToDo: implement your own tool
    pass

tools = [
    # your tools here
]

SYSTEM_PROMPT = """YOUR SYSTEM PROMTP HERE"""

config = {
    "model": "qwen3:8b",
    "base_url": "http://127.0.0.1:11434/v1",
	"api_key": "ollama",
    "max_tokens": 8192,
    "temperature": 0,
    "seed": 42,
}

model = ChatOpenAI(
    **config
)
react_agent = create_agent(
    model,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)

## Home Exercise Build your own RAG agent here

(maybe you can reuse the stuff from last session)

In [ ]:
@tool
def retrieve_documents(query: str):
    # ToDo: implement your own tool
    # here needs to be all the langchain stuff from last session
    # maybe you can even implement re-ranking
    return "No documents found"

tools = [
    retrieve_documents
]

SYSTEM_PROMPT = """You are a knowledgeable and helpful assistant that answers user queries.
You are able to retrieve and accurate information from a documents storage."""

config = {
    "model": "qwen3:8b",
    "base_url": "http://127.0.0.1:11434/v1",
	"api_key": "ollama",
    "max_tokens": 8192,
    "temperature": 0,
    "seed": 42,
}

model = ChatOpenAI(
    **config
)
react_agent = create_agent(
    model,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)

## A Multi-Agent System

Multi-agent systems combine multiple usually specialized agents ([Guo et al., 2024](https://doi.org/10.48550/arXiv.2402.01680)). These systems, as visualized in this graph from Guo et al. (2024) can be distinguished by their communication strategy:

![image.png](https://github.com/olivermueller/amlta-2025/blob/main/Session_06/imgs/multi_agent_systems_communication.png?raw=true)

(Adapted from [Guo et al., 2024](https://doi.org/10.48550/arXiv.2402.01680))

Usually centralized architectures are most simply to implement, as agents can simply be incorporated as tools for a controller agent. For our basics session, this will be the multi-agent system architecture to go.

In LangChain, this can be solved via [ToolCalling](https://docs.langchain.com/oss/python/langchain/multi-agent#tool-calling)

Here is the description from the LangChain Documentation ([Langchain, 2025]((https://docs.langchain.com/oss/python/langchain/multi-agent#tool-calling))):

"In tool calling, one agent (the “controller”) treats other agents as tools to be invoked when needed. The controller manages orchestration, while tool agents perform specific tasks and return results.
Flow:"
- The controller receives input and decides which tool (subagent) to call.
- The tool agent runs its task based on the controller’s instructions.
- The tool agent returns results to the controller.
- The controller decides the next step or finishes.

![image](https://github.com/olivermueller/amlta-2025/blob/main/Session_06/imgs/langchain.png?raw=true)

Let's implement this with two sub agents. A maths agent and a retriever agent to get information from wikipedia.

In [ ]:
from langchain.tools import tool
from langchain.agents import create_agent

First lets define our wikipedia agent.

In [11]:
@tool
def search_in_wikipedia(query: str) -> str:
    """Search Wikipedia for a given query."""
    retriever = WikipediaRetriever()
    docs = retriever.invoke(query)
    results = "\n\n-----\n\n".join([f"Document {i}:\n\nMetadata:\n-Title: {docs[i].metadata['title'].strip()}\n-Path: https://en.wikipedia.org/wiki/{docs[i].metadata['title'].strip().replace(' ', '_')}\n\nContent:\n{docs[i].metadata['summary'].strip()}" for i in range(len(docs))])

    return results

tools = [
    search_in_wikipedia
]

SYSTEM_PROMPT = """You are a knowledgeable and helpful assistant that answers user queries.
You are able to retrieve and accurate information from Wikipedia."""

config = {
    "model": "qwen3:8b",
    "base_url": "http://127.0.0.1:11434/v1",
	"api_key": "ollama",
    "max_tokens": 8192,
    "temperature": 0,
    "seed": 42,
}

model = ChatOpenAI(
    **config
)
retriever_agent = create_agent(
    model,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)

Then the same for the math agent.

In [12]:
@tool
def add_numbers(a: float, b: float) -> float:
    """Add two numbers together."""
    return a + b

@tool
def multiply_numbers(a: float, b: float) -> float:
    """Multiply two numbers together."""
    return a * b

@tool
def divide_numbers(a: float, b: float) -> float:
    """Divide the first number by the second number."""
    return a / b

@tool
def subtract_numbers(a: float, b: float) -> float:
    """Subtract the second number from the first number."""
    return a - b

tools = [
    add_numbers,
    multiply_numbers,
    divide_numbers,
    subtract_numbers
]

SYSTEM_PROMPT = """You are a knowledgeable and helpful assistant that answers math questions.
You are able to use basic arithmetic tools."""

config = {
    "model": "qwen3:8b",
    "base_url": "http://127.0.0.1:11434/v1",
	"api_key": "ollama",
    "max_tokens": 8192,
    "temperature": 0,
    "seed": 42,
}

model = ChatOpenAI(
    **config
)
math_agent = create_agent(
    model,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)

Now we are simply building a third agent that is deciding which agent to call.

In [13]:
@tool(
    "retriever_agent",
    description="Agent that retrieves information from wikipedia."
)
def call_retriever_agent(query: str):
    result = retriever_agent.invoke({
        "messages": [{"role": "user", "content": query}]
    })
    return result["messages"][-1].content

@tool(
    "math_agent",
    description="Agent that performs basic math operations."
)
def call_math_agent(query: str):
    result = math_agent.invoke({
        "messages": [{"role": "user", "content": query}]
    })
    return result["messages"][-1].content

tools = [
    call_retriever_agent,
    call_math_agent
]

SYSTEM_PROMPT = """You are a knowledgeable and helpful assistant that controlls two other agents.
There is one agent that can do maths and the other retrieves and synthesizes information for you.
Call the appropriate agent, when you are prompted by the user and answer based on the agent's answer."""

config = {
    "model": "qwen3:8b",
    "base_url": "http://127.0.0.1:11434/v1",
	"api_key": "ollama",
    "max_tokens": 8192,
    "temperature": 0,
    "seed": 42,
}

model = ChatOpenAI(
    **config
)

agent = create_agent(
    model,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)

Wow that was easy, right? Let's try it out.

In [17]:
query = "Who is Daenerys Targaryen and who is her actress?"
messages = [HumanMessage(content=query)]
messages = agent.invoke({"messages": messages})["messages"]
for message in messages:
    print(message.pretty_repr())

================================ Human Message =================================

Who is Daenerys Targaryen and who is her actress?
================================== Ai Message ==================================
Tool Calls:
  retriever_agent (call_l6n9mmyq)
 Call ID: call_l6n9mmyq
  Args:
    query: Daenerys Targaryen
  retriever_agent (call_q8o45ad6)
 Call ID: call_q8o45ad6
  Args:
    query: Daenerys Targaryen actress
================================= Tool Message =================================
Name: retriever_agent

Daenerys Targaryen is a central character in **George R. R. Martin's** *A Song of Ice and Fire* series and its television adaptation *Game of Thrones*. Here's a concise overview based on the Wikipedia entry:

### **Background & Story Arc**
- **Origin**: Introduced in *A Game of Thrones* (1996), Daenerys is the last surviving heir of the deposed Targaryen dynasty, which once ruled Westeros. She is exiled in Essos with her brother Viserys, who seeks to reclaim the Iron

In [20]:
query = "What is 1000 x 734 / 437?"
messages = [HumanMessage(content=query)]
messages = agent.invoke({"messages": messages})["messages"]
for message in messages:
    print(message.pretty_repr())

================================ Human Message =================================

What is 1000 x 734 / 437?
================================== Ai Message ==================================
Tool Calls:
  math_agent (call_re21g3pt)
 Call ID: call_re21g3pt
  Args:
    query: 1000 x 734 / 437
================================= Tool Message =================================
Name: math_agent

The result of $1000 \times 734 \div 437$ is approximately **1679.63**.
================================== Ai Message ==================================

The result of $1000 \times 734 \div 437$ is approximately **1679.63**. 

Here's the breakdown:
1. Multiply 1000 by 734: $1000 \times 734 = 734{,}000$
2. Divide by 437: $734{,}000 \div 437 \approx 1679.63$


Wow this worked. Now let's look into some real-world that used agents.